In [63]:
import os

BASE_DIR = os.getcwd()
DATASET_PATH = os.path.join(BASE_DIR, "datasets")

In [64]:
import pandas as pd
import os

DATASET_PATH = r"C:\Users\Arpita\OneDrive\Desktop\AgroAssist 2.0\datasets"

crop_df = pd.read_excel(os.path.join(DATASET_PATH, "crop_dataset.xlsx"))
tree_df = pd.read_excel(os.path.join(DATASET_PATH, "trees_dataset.xlsx"))
flower_df = pd.read_excel(os.path.join(DATASET_PATH, "flower_dataset.xlsx"))
mushroom_df = pd.read_excel(os.path.join(DATASET_PATH, "mushroom_dataset.xlsx"))
bee_df = pd.read_excel(os.path.join(DATASET_PATH, "bee_dataset.xlsx"))
dairy_df = pd.read_excel(os.path.join(DATASET_PATH, "dairy_dataset.xlsx"))
livestock_df = pd.read_excel(os.path.join(DATASET_PATH, "livestock_dataset.xlsx"))

print(" All datasets loaded")

 All datasets loaded


In [65]:
import os
print(os.listdir(DATASET_PATH))

['bee_dataset.xlsx', 'crop_dataset.xlsx', 'dairy_dataset.xlsx', 'flower_dataset.xlsx', 'livestock_dataset.xlsx', 'mushroom_dataset.xlsx', 'trees_dataset.xlsx']


In [66]:
print(tree_df.columns)

Index(['Zone', 'State', 'District', 'Tree_Type', 'Category', 'Season',
       'Cost_per_acre', 'Revenue_per_acre', 'Latitude', 'Longitude',
       'Soil_Type', 'Rainfall_mm', 'Temperature_C', 'Water_Availability',
       'Profit_per_acre'],
      dtype='object')


In [67]:
print("Tree:", tree_df.columns)
print("Flower:", flower_df.columns)
print("Dairy:", dairy_df.columns)
print("Mushroom:", mushroom_df.columns)
print("Bee:", bee_df.columns)
print("Livestock:", livestock_df.columns)

Tree: Index(['Zone', 'State', 'District', 'Tree_Type', 'Category', 'Season',
       'Cost_per_acre', 'Revenue_per_acre', 'Latitude', 'Longitude',
       'Soil_Type', 'Rainfall_mm', 'Temperature_C', 'Water_Availability',
       'Profit_per_acre'],
      dtype='object')
Flower: Index(['Zone', 'State', 'District', 'Latitude', 'Longitude', 'Crop',
       'Market_Segment', 'Season', 'Water_Availability', 'Soil_Type',
       'Soil_pH', 'Rainfall_mm', 'Temperature_C', 'Farming_Type',
       'Expected_Yield_ton_per_acre', 'Market_Price_per_ton',
       'Profit_per_acre', 'Allied_Activity', 'Profit_Level'],
      dtype='object')
Dairy: Index(['Animal_Breed', 'Animal_Type', 'Climate_Suitability', 'Temp_Min_C',
       'Temp_Max_C', 'Feed_Type', 'Water_Requirement',
       'Avg_Milk_Liters_Per_Day', 'Lactation_Period_Days',
       'Disease_Resistance', 'Maintenance_Cost', 'Profit_Level'],
      dtype='object')
Mushroom: Index(['Name', 'Type', 'Category', 'Climate_Suitability', 'Purpose',
       'A

In [68]:
TREE_COL = "Tree_Type"
FLOWER_COL = "Crop"              # flowers dataset uses 'Crop'
DAIRY_COL = "Animal_Breed"
MUSHROOM_COL = "Name"
BEE_COL = "Breed"
LIVESTOCK_COL = "Breed"

In [128]:
import random

def create_ifs_dataset():
    data = []
    for _, crop in crop_df.iterrows():
        for _ in range(8):  # Create multiple variations for each crop entry
            row = {}
            # --- INPUT FEATURES ---
            row["Zone"] = crop["Zone"]
            row["Soil_pH"] = crop["Soil_pH"]
            row["Rainfall"] = crop["Rainfall_mm"]
            row["Temperature"] = crop["Temperature_C"]
            row["Season"] = crop["Season"]
            row["Budget"] = crop["Profit_per_acre"]
            
            # --- OUTPUTS (TARGETS) ---
            row["Crop"] = crop["Crop"]
            
            # 1. SMART TREE: Based on Weather (Temperature & Rainfall)
            tree_filtered = tree_df[
                (tree_df["Temperature_C"].between(crop["Temperature_C"] - 5, crop["Temperature_C"] + 5)) &
                (tree_df["Rainfall_mm"].between(crop["Rainfall_mm"] - 200, crop["Rainfall_mm"] + 200))
            ]
            row["Tree"] = tree_filtered.sample(1)[TREE_COL].values[0] if not tree_filtered.empty else tree_df.sample(1)[TREE_COL].values[0]
            
            # 2. FLOWER: Random sampling (can be refined based on Soil later)
            row["Flower"] = flower_df.sample(1)[FLOWER_COL].values[0]
            
            # 3. SMART DAIRY: Based on Temperature tolerance
            dairy_filtered = dairy_df[
                (dairy_df["Temp_Max_C"] >= crop["Temperature_C"]) & 
                (dairy_df["Temp_Min_C"] <= crop["Temperature_C"])
            ]
            row["Dairy"] = dairy_filtered.sample(1)[DAIRY_COL].values[0] if not dairy_filtered.empty else dairy_df.sample(1)[DAIRY_COL].values[0]
            
            # 4. SMART LIVESTOCK: Parsing the temperature range string (e.g., "15-35")
            def is_livestock_suitable(temp_range_str, target_temp):
                try:
                    # Splits "15-35" into 15 and 35
                    low, high = map(int, str(temp_range_str).split('-'))
                    return low <= target_temp <= high
                except:
                    return True # Default to true if format is unexpected
            
            # Filter livestock based on the helper function above
            livestock_suitable_indices = [
                idx for idx, l_row in livestock_df.iterrows() 
                if is_livestock_suitable(l_row["Temperature_Range_C"], crop["Temperature_C"])
            ]
            
            if livestock_suitable_indices:
                row["Livestock"] = livestock_df.loc[random.choice(livestock_suitable_indices), LIVESTOCK_COL]
            else:
                row["Livestock"] = livestock_df.sample(1)[LIVESTOCK_COL].values[0]

            # 5. MUSHROOM LOGIC: Link to specific Crop Byproducts
            if crop["Byproduct"] in ["Straw", "Bagasse", "Husk"]:
                row["Mushroom"] = mushroom_df.sample(1)[MUSHROOM_COL].values[0]
            else:
                row["Mushroom"] = "None"
            
            # 6. BEE LOGIC: Link to Flower Pollination
            if row["Flower"] in ["Rose", "Marigold", "Jasmine"]:
                row["Bee"] = bee_df.sample(1)[BEE_COL].values[0]
            else:
                row["Bee"] = "None"
                
            data.append(row)
    return pd.DataFrame(data)

# Re-run the dataset generation with logical filtering
ifs_df = create_ifs_dataset()

print(" IFS dataset updated with Climate & Byproduct logic.")
print(f"Total synthetic training rows: {len(ifs_df)}")
print(ifs_df.head())


 IFS dataset updated with Climate & Byproduct logic.
Total synthetic training rows: 960
             Zone  Soil_pH  Rainfall  Temperature  Season  Budget       Crop  \
0  Upper_Gangetic      7.2       850           28       1   89600  Sugarcane   
1  Upper_Gangetic      7.2       850           28       1   89600  Sugarcane   
2  Upper_Gangetic      7.2       850           28       1   89600  Sugarcane   
3  Upper_Gangetic      7.2       850           28       1   89600  Sugarcane   
4  Upper_Gangetic      7.2       850           28       1   89600  Sugarcane   

         Tree     Flower              Dairy          Livestock  \
0       Guava   Marigold         Jamunapari  New Zealand White   
1       Mango  Gladiolus         Red Sindhi           Vanaraja   
2  Eucalyptus       Rose            Mehsana           Ghungroo   
3       Guava  Gladiolus                Gir  New Zealand White   
4      Poplar    Jasmine  Holstein Friesian       Bengal Local   

          Mushroom                

In [70]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

for col in ifs_df.columns:
    le = LabelEncoder()
    ifs_df[col] = le.fit_transform(ifs_df[col].astype(str))
    encoders[col] = le

print(" Encoding done")

 Encoding done


In [71]:
print(encoders["Zone"].classes_)

['Central_Plateau' 'Delhi_Region' 'Lower_Gangetic' 'Middle_Gangetic'
 'Trans_Gangetic' 'Upper_Gangetic']


In [72]:
ifs_df = create_ifs_dataset()

In [73]:
ifs_df = create_ifs_dataset()

In [74]:
from sklearn.preprocessing import LabelEncoder

encoded_df = ifs_df.copy()
encoders = {}

# Only categorical columns
categorical_cols = ["Zone", "Crop", "Tree", "Flower", "Dairy", "Mushroom", "Bee", "Livestock"]

for col in categorical_cols:
    le = LabelEncoder()
    encoded_df[col] = le.fit_transform(encoded_df[col].astype(str))
    encoders[col] = le

print(" Correct encoding done")

 Correct encoding done


In [75]:
feature_cols = ["Zone", "Soil_pH", "Rainfall", "Temperature", "Season", "Budget"]
target_cols = ["Crop", "Tree", "Flower", "Dairy", "Mushroom", "Bee", "Livestock"]

X = encoded_df[feature_cols]
y = encoded_df[target_cols]

from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

model = MultiOutputClassifier(RandomForestClassifier(n_estimators=100))
model.fit(X, y)

print("Model trained")

Model trained


In [76]:
sample_input = pd.DataFrame([{
    "Zone": "Upper_Gangetic",
    "Soil_pH": 7.2,   # ⚠️ use realistic value from dataset
    "Rainfall": 850,
    "Temperature": 28,
    "Season": 1,
    "Budget": 50000
}])

In [77]:
sample_input["Zone"] = encoders["Zone"].transform(sample_input["Zone"])

In [78]:
pred = model.predict(sample_input)[0]

result = {}
for i, col in enumerate(target_cols):
    result[col] = encoders[col].inverse_transform([pred[i]])[0]

print("Recommended IFS Plan:")
print(result)

Recommended IFS Plan:
{'Crop': 'Rice', 'Tree': 'Poplar', 'Flower': 'Jasmine', 'Dairy': 'Surti', 'Mushroom': 'None', 'Bee': 'None', 'Livestock': 'Ghungroo'}


In [79]:
print(ifs_df["Dairy"].unique())
print(ifs_df["Livestock"].unique())

['Murrah' 'Sirohi' 'Bhadawari' 'Barbari' 'Jaffarabadi' 'Surti'
 'Red Sindhi' 'Gir' 'Mehsana' 'Jersey' 'Tharparkar' 'Sahiwal'
 'Holstein Friesian' 'Jamunapari' 'Nili Ravi' 'Beetal']
['Kadaknath' 'Ghungroo' 'Khaki Campbell' 'Beetal' 'Landrace'
 'Large White Fork' 'Indian Runner' 'Duroc' 'Broad Breasted White'
 'Vanaraja' 'Hampshire' 'Niang Megha' 'Japanese Quail' 'Soviet Chinchilla'
 'Bengal Local' 'Rhode Island Red' 'Barbari' 'White Leghorn'
 'Black Bengal' 'Sirohi' 'New Zealand White' 'Plymouth Rock' 'Jamunapari'
 'Grey Giant']


In [80]:
def get_top_k_predictions(model, sample_input, k=3):
    probas = model.predict_proba(sample_input)

    top_preds = {}

    for i, col in enumerate(target_cols):
        probs = probas[i][0]  # probabilities for this output
        classes = encoders[col].classes_

        # Get top k indices
        top_k_idx = probs.argsort()[-k:][::-1]

        top_preds[col] = [(classes[idx], probs[idx]) for idx in top_k_idx]

    return top_preds

In [81]:
import itertools

def generate_ifs_strategies(top_preds):
    strategies = []

    for combo in itertools.product(
        top_preds["Crop"],
        top_preds["Tree"],
        top_preds["Flower"],
        top_preds["Dairy"],
        top_preds["Mushroom"],
        top_preds["Bee"],
        top_preds["Livestock"]
    ):
        strategy = {
            "Crop": combo[0][0],
            "Tree": combo[1][0],
            "Flower": combo[2][0],
            "Dairy": combo[3][0],
            "Mushroom": combo[4][0],
            "Bee": combo[5][0],
            "Livestock": combo[6][0],
        }

        # Base score = sum of probabilities
        score = sum([
            combo[0][1],
            combo[1][1],
            combo[2][1],
            combo[3][1],
            combo[4][1],
            combo[5][1],
            combo[6][1]
        ])

        strategy["score"] = score
        strategies.append(strategy)

    return strategies

In [82]:
def apply_suitability_rules(strategy):
    score = strategy["score"]

    # Flower → Bee dependency
    if strategy["Flower"] in ["Rose", "Marigold", "Jasmine"]:
        if strategy["Bee"] != "None":
            score += 2
        else:
            score -= 2

    #  Crop byproduct → Mushroom
    if strategy["Crop"] in ["Sugarcane", "Wheat"]:
        if strategy["Mushroom"] != "None":
            score += 2

    #  Dairy + Livestock overload penalty
    if strategy["Dairy"] != "None" and strategy["Livestock"] != "None":
        score -= 1

    return score

In [83]:
sample_input = pd.DataFrame([{
    "Zone": "Upper_Gangetic",
    "Soil_pH": 7.2,
    "Rainfall": 850,
    "Temperature": 28,
    "Season": 1,
    "Budget": 50000
}])

top_strategies = get_best_ifs_strategies(sample_input)

for i, s in enumerate(top_strategies):
    print(f"\nStrategy {i+1}:")
    print(s)


Strategy 1:
{'Crop': 'Sugarcane', 'Tree': 'Poplar', 'Flower': 'Jasmine', 'Dairy': 'Surti', 'Mushroom': 'Oyster Mushroom', 'Bee': 'Apis mellifera', 'Livestock': 'Ghungroo', 'score': np.float64(1.5355524723814196), 'final_score': np.float64(4.535552472381419)}

Strategy 2:
{'Crop': 'Sugarcane', 'Tree': 'Guava', 'Flower': 'Jasmine', 'Dairy': 'Murrah', 'Mushroom': 'Oyster Mushroom', 'Bee': 'Apis cerana indica', 'Livestock': 'Ghungroo', 'score': np.float64(1.5091002301791774), 'final_score': np.float64(4.509100230179177)}

Strategy 3:
{'Crop': 'Sugarcane', 'Tree': 'Guava', 'Flower': 'Jasmine', 'Dairy': 'Sirohi', 'Mushroom': 'Oyster Mushroom', 'Bee': 'Apis mellifera', 'Livestock': 'Kadaknath', 'score': np.float64(1.491758766087713), 'final_score': np.float64(4.491758766087713)}


In [84]:
def is_similar(s1, s2):
    same = 0
    for key in ["Crop", "Tree", "Flower", "Dairy", "Mushroom", "Bee", "Livestock"]:
        if s1[key] == s2[key]:
            same += 1
    return same >= 5   # if too similar → reject

In [85]:
def get_diverse_strategies(strategies, top_n=3):
    final = []

    for s in strategies:
        if all(not is_similar(s, existing) for existing in final):
            final.append(s)
        if len(final) == top_n:
            break

    return final

In [86]:
def get_best_ifs_strategies(sample_input, top_n=3):

    # Encode only if not already encoded
    if isinstance(sample_input["Zone"].iloc[0], str):
        sample_input["Zone"] = encoders["Zone"].transform(sample_input["Zone"])

    top_preds = get_top_k_predictions(model, sample_input, k=3)

    strategies = generate_ifs_strategies(top_preds)

    for s in strategies:
        s["final_score"] = apply_suitability_rules(s)

    strategies = sorted(strategies, key=lambda x: x["final_score"], reverse=True)

    return get_diverse_strategies(strategies, top_n)

In [87]:
top_strategies = get_best_ifs_strategies(sample_input)

for i, s in enumerate(top_strategies):
    print(f"\nStrategy {i+1}")
    print(s)


Strategy 1
{'Crop': 'Sugarcane', 'Tree': 'Poplar', 'Flower': 'Jasmine', 'Dairy': 'Surti', 'Mushroom': 'Oyster Mushroom', 'Bee': 'Apis mellifera', 'Livestock': 'Ghungroo', 'score': np.float64(1.5355524723814196), 'final_score': np.float64(4.535552472381419)}

Strategy 2
{'Crop': 'Sugarcane', 'Tree': 'Guava', 'Flower': 'Jasmine', 'Dairy': 'Murrah', 'Mushroom': 'Oyster Mushroom', 'Bee': 'Apis cerana indica', 'Livestock': 'Ghungroo', 'score': np.float64(1.5091002301791774), 'final_score': np.float64(4.509100230179177)}

Strategy 3
{'Crop': 'Sugarcane', 'Tree': 'Guava', 'Flower': 'Jasmine', 'Dairy': 'Sirohi', 'Mushroom': 'Oyster Mushroom', 'Bee': 'Apis mellifera', 'Livestock': 'Kadaknath', 'score': np.float64(1.491758766087713), 'final_score': np.float64(4.491758766087713)}


In [88]:
for i, s in enumerate(top_strategies):
    print(f"\nStrategy {i+1}")
    print("-" * 30)

    for key in ["Crop", "Tree", "Flower", "Dairy", "Mushroom", "Bee", "Livestock"]:
        print(f"{key}: {s[key]}")

    print(f"Suitability Score: {round(float(s['final_score']), 2)}")


Strategy 1
------------------------------
Crop: Sugarcane
Tree: Poplar
Flower: Jasmine
Dairy: Surti
Mushroom: Oyster Mushroom
Bee: Apis mellifera
Livestock: Ghungroo
Suitability Score: 4.54

Strategy 2
------------------------------
Crop: Sugarcane
Tree: Guava
Flower: Jasmine
Dairy: Murrah
Mushroom: Oyster Mushroom
Bee: Apis cerana indica
Livestock: Ghungroo
Suitability Score: 4.51

Strategy 3
------------------------------
Crop: Sugarcane
Tree: Guava
Flower: Jasmine
Dairy: Sirohi
Mushroom: Oyster Mushroom
Bee: Apis mellifera
Livestock: Kadaknath
Suitability Score: 4.49


In [89]:
def explain_strategy(s):
    explanations = []

    if s["Crop"] == "Sugarcane":
        explanations.append("Sugarcane provides bagasse → supports mushroom cultivation")

    if s["Flower"] in ["Rose", "Marigold"]:
        explanations.append("Flower crops support beekeeping → improves pollination & income")

    if s["Dairy"] != "None":
        explanations.append("Dairy provides manure → useful for biogas and organic farming")

    return explanations

In [90]:
for i, s in enumerate(top_strategies):
    print(f"\n Strategy {i+1}")
    print("-" * 30)

    for key in ["Crop", "Tree", "Flower", "Dairy", "Mushroom", "Bee", "Livestock"]:
        print(f"{key}: {s[key]}")

    print(f"Suitability Score: {round(float(s['final_score']), 2)}")

    print(" Why this works:")
    for exp in explain_strategy(s):
        print("-", exp)


 Strategy 1
------------------------------
Crop: Sugarcane
Tree: Poplar
Flower: Jasmine
Dairy: Surti
Mushroom: Oyster Mushroom
Bee: Apis mellifera
Livestock: Ghungroo
Suitability Score: 4.54
 Why this works:
- Sugarcane provides bagasse → supports mushroom cultivation
- Dairy provides manure → useful for biogas and organic farming

 Strategy 2
------------------------------
Crop: Sugarcane
Tree: Guava
Flower: Jasmine
Dairy: Murrah
Mushroom: Oyster Mushroom
Bee: Apis cerana indica
Livestock: Ghungroo
Suitability Score: 4.51
 Why this works:
- Sugarcane provides bagasse → supports mushroom cultivation
- Dairy provides manure → useful for biogas and organic farming

 Strategy 3
------------------------------
Crop: Sugarcane
Tree: Guava
Flower: Jasmine
Dairy: Sirohi
Mushroom: Oyster Mushroom
Bee: Apis mellifera
Livestock: Kadaknath
Suitability Score: 4.49
 Why this works:
- Sugarcane provides bagasse → supports mushroom cultivation
- Dairy provides manure → useful for biogas and organic f

In [91]:
def add_subsidy_info(s):
    if s["Bee"] != "None":
        print(" Eligible for National Beekeeping & Honey Mission subsidy")

    if s["Dairy"] != "None":
        print(" Eligible for Dairy Entrepreneurship Development Scheme")

In [92]:
def add_system_suggestions(s):
    print("Water Management: Drip irrigation recommended")

    if s["Dairy"] != "None":
        print(" Biogas Plant recommended using animal waste")

In [93]:
def add_system_suggestions(s, sample_input):
    print(" Water Management:")

    rainfall = sample_input["Rainfall"].iloc[0]
    crop = s["Crop"]

    #  Rainfall-based logic
    if rainfall < 700:
        print("- Low rainfall → Drip irrigation recommended")
        print("- Rainwater harvesting advised")
    elif 700 <= rainfall <= 1000:
        print("- Moderate rainfall → Sprinkler irrigation suitable")
    else:
        print("- High rainfall → Ensure proper drainage system")

    #  Crop-specific suggestion
    if crop == "Sugarcane":
        print("- Sugarcane is water intensive → use drip irrigation for efficiency")

    #  Biogas logic
    if s["Dairy"] != "None":
        print(" - Biogas Plant:")
        print("- Animal waste can be converted into biogas")
        print("- Slurry can be used as organic fertilizer")

In [94]:
for i, s in enumerate(top_strategies):
    print(f"\n Strategy {i+1}")
    print("-" * 30)

    for key in ["Crop", "Tree", "Flower", "Dairy", "Mushroom", "Bee", "Livestock"]:
        print(f"{key}: {s[key]}")

    print(f"Suitability Score: {round(float(s['final_score']), 2)}")

    print("\n System Suggestions:")
    add_system_suggestions(s, sample_input)


 Strategy 1
------------------------------
Crop: Sugarcane
Tree: Poplar
Flower: Jasmine
Dairy: Surti
Mushroom: Oyster Mushroom
Bee: Apis mellifera
Livestock: Ghungroo
Suitability Score: 4.54

 System Suggestions:
 Water Management:
- Moderate rainfall → Sprinkler irrigation suitable
- Sugarcane is water intensive → use drip irrigation for efficiency
 - Biogas Plant:
- Animal waste can be converted into biogas
- Slurry can be used as organic fertilizer

 Strategy 2
------------------------------
Crop: Sugarcane
Tree: Guava
Flower: Jasmine
Dairy: Murrah
Mushroom: Oyster Mushroom
Bee: Apis cerana indica
Livestock: Ghungroo
Suitability Score: 4.51

 System Suggestions:
 Water Management:
- Moderate rainfall → Sprinkler irrigation suitable
- Sugarcane is water intensive → use drip irrigation for efficiency
 - Biogas Plant:
- Animal waste can be converted into biogas
- Slurry can be used as organic fertilizer

 Strategy 3
------------------------------
Crop: Sugarcane
Tree: Guava
Flower: Ja

In [127]:
from sklearn.neighbors import KNeighborsClassifier

# Train the Zone Detector using your existing crop_df
# This maps Latitude/Longitude directly to the 'Zone' labels in your data
X_zone = crop_df[['Latitude', 'Longitude']]
y_zone = crop_df['Zone']

zone_knn = KNeighborsClassifier(n_neighbors=3)
zone_knn.fit(X_zone, y_zone)

def get_zone_from_location(lat, lon):
    # Use the trained KNN model to predict the zone for any new coordinates
    predicted_zone = zone_knn.predict([[lat, lon]])[0]
    return predicted_zone

print("KNN Zone Detector trained successfully.")

KNN Zone Detector trained successfully.


In [96]:
pip install geopy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [97]:
from geopy.geocoders import Nominatim

def get_coordinates(place_name):
    geolocator = Nominatim(user_agent="agro_assist")

    location = geolocator.geocode(place_name)

    if location:
        return location.latitude, location.longitude
    else:
        return None, None

In [98]:
place = input("Enter your district/city: ")

lat, lon = get_coordinates(place)

if lat is None:
    print("Location not found")
else:
    print("Latitude:", lat)
    print("Longitude:", lon)

Enter your district/city:  meerut


Latitude: 29.0018557
Longitude: 77.7679671


In [99]:
zone = get_zone_from_location(lat, lon)

sample_input = pd.DataFrame([{
    "Zone": zone,
    "Soil_pH": 7.2,
    "Rainfall": 850,
    "Temperature": 28,
    "Season": 1,
    "Budget": 50000
}])

In [100]:
def preprocess_live_data(temp, rainfall):
    
    # Clamp values to dataset range
    temp = max(10, min(temp, 45))
    rainfall = max(100, min(rainfall, 2000))
    
    return temp, rainfall

In [101]:
def process_forecast(forecast_data):
    temps = [day["temp"] for day in forecast_data]
    avg_temp = sum(temps) / len(temps)
    return avg_temp

In [115]:
import requests

API_KEY = "your_real_api_key_here"

def get_weather(lat, lon):
    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "lat": lat,
        "lon": lon,
        "appid": API_KEY,
        "units": "metric"
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print("API Error:", response.status_code)
        return 28, 60, 0   # fallback

    data = response.json()

    # Debug once
    print("Weather API:", data)

    if "main" not in data:
        print("Invalid response:", data)
        return 28, 60, 0

    temp = data["main"]["temp"]
    humidity = data["main"]["humidity"]
    rainfall = data.get("rain", {}).get("1h", 0)

    return temp, humidity, rainfall

In [116]:
def preprocess_live_data(temp, rainfall):
    # You can smooth/extreme-handle values
    if rainfall is None:
        rainfall = 0
    return temp, rainfall

In [118]:
place = input("Enter your district/city: ")
budget = int(input("Enter your budget: "))
season = int(input("Enter season (1=Rabi, 2=Kharif): "))

lat, lon = get_coordinates(place)

if lat is None:
    print("Location not found")
else:
    zone = get_zone_from_location(lat, lon)

    temp, humidity, rainfall = get_weather(lat, lon)
    temp, rainfall = preprocess_live_data(temp, rainfall)

    sample_input = pd.DataFrame([{
        "Zone": zone,
        "Soil_pH": 7.2,  # later make dynamic
        "Rainfall": rainfall,
        "Temperature": temp,
        "Season": season,
        "Budget": budget
    }])

    strategies = get_best_ifs_strategies(sample_input)

    for i, s in enumerate(strategies):
        print(f"\nStrategy {i+1}")
        for key in ["Crop","Tree","Flower","Dairy","Mushroom","Bee","Livestock"]:
            print(f"{key}: {s[key]}")

Enter your district/city:  panipat
Enter your budget:  40100
Enter season (1=Rabi, 2=Kharif):  1


API Error: 401

Strategy 1
Crop: Rice
Tree: Guava
Flower: Marigold
Dairy: Jamunapari
Mushroom: None
Bee: Apis mellifera
Livestock: Ghungroo

Strategy 2
Crop: Rice
Tree: Guava
Flower: Jasmine
Dairy: Mehsana
Mushroom: None
Bee: Apis mellifera
Livestock: Sirohi

Strategy 3
Crop: Rice
Tree: Eucalyptus
Flower: Marigold
Dairy: Mehsana
Mushroom: None
Bee: Apis mellifera
Livestock: Khaki Campbell


In [119]:
def get_weather_forecast(lat, lon):
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&daily=temperature_2m_max,temperature_2m_min,precipitation_sum&timezone=auto"

    response = requests.get(url)
    data = response.json()

    temps_max = data["daily"]["temperature_2m_max"]
    temps_min = data["daily"]["temperature_2m_min"]
    rainfall = data["daily"]["precipitation_sum"]

    # ✅ 7-day averages
    avg_temp = sum([(temps_max[i] + temps_min[i]) / 2 for i in range(7)]) / 7
    total_rainfall = sum(rainfall[:7])

    return avg_temp, total_rainfall

In [120]:
def preprocess_weather(avg_temp, rainfall):
    
    # Rainfall normalization
    if rainfall < 20:
        rainfall_level = 300   # dry
    elif rainfall < 60:
        rainfall_level = 700   # moderate
    else:
        rainfall_level = 1200  # high

    # Temperature smoothing
    if avg_temp < 20:
        temp = 18
    elif avg_temp < 30:
        temp = 26
    else:
        temp = 32

    return temp, rainfall_level

In [122]:
def predict_ifs(place, budget, season):

    lat, lon = get_coordinates(place)

    if lat is None:
        return {"error": "Location not found"}

    zone = get_zone_from_location(lat, lon)

    #  7-day forecast
    avg_temp, total_rainfall = get_weather_forecast(lat, lon)

    #  preprocess for model compatibility
    temp, rainfall = preprocess_weather(avg_temp, total_rainfall)

    sample_input = pd.DataFrame([{
        "Zone": zone,
        "Soil_pH": 7.2,   # later you can improve this
        "Rainfall": rainfall,
        "Temperature": temp,
        "Season": season,
        "Budget": budget
    }])

    strategies = get_best_ifs_strategies(sample_input)

    return strategies

In [123]:
def weather_summary(avg_temp, rainfall):
    if rainfall < 20:
        return "Low rainfall expected → irrigation needed"
    elif rainfall < 60:
        return "Moderate rainfall → good for most crops"
    else:
        return "High rainfall → ensure drainage system"